# Healthcare Cost Driver Analysis

## Project Overview
Analyze healthcare billing and patient cost data to identify major drivers of treatment cost and cost variation across patient demographics, diagnoses, and facility types.

## Learning Objectives
- Identify key factors driving healthcare costs
- Analyze cost variation by diagnosis, procedure type, and demographics
- Detect outlier charges and cost inefficiencies
- Quantify the relative impact of different cost drivers

## Problem Statement
Healthcare administrators need to understand what drives cost variation to identify savings opportunities, improve pricing transparency, and allocate resources efficiently.

## Why This Project Matters
Healthcare costs are a major economic burden. Understanding cost drivers enables targeted interventions, better budgeting, and more equitable pricing.

## Dataset Overview
Synthetic healthcare billing dataset: ~8,000 patient encounters with charges, diagnoses, length of stay, insurance type, and demographics.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 8000
age = np.clip(np.random.normal(52, 20, n).astype(int), 1, 95)
gender = np.random.choice(['Male', 'Female'], n, p=[0.48, 0.52])
insurance = np.random.choice(['Private', 'Medicare', 'Medicaid', 'Self-Pay'], n, p=[0.40, 0.30, 0.20, 0.10])
diagnosis = np.random.choice(['Heart Disease', 'Diabetes', 'Respiratory', 'Orthopedic',
                                'Cancer', 'GI Disorder', 'Mental Health', 'Other'],
                               n, p=[0.12, 0.15, 0.13, 0.14, 0.08, 0.12, 0.10, 0.16])
procedure = np.random.choice(['Surgery', 'Imaging', 'Lab Work', 'Therapy', 'Emergency', 'Routine'],
                               n, p=[0.15, 0.18, 0.22, 0.15, 0.12, 0.18])
los = np.clip(np.random.exponential(3.5, n).astype(int), 0, 30)
# Cost driven by diagnosis, procedure, and LOS
base_cost = {'Heart Disease': 15000, 'Cancer': 20000, 'Orthopedic': 12000, 'Respiratory': 8000,
             'Diabetes': 6000, 'GI Disorder': 7000, 'Mental Health': 5000, 'Other': 4000}
proc_mult = {'Surgery': 2.5, 'Emergency': 1.8, 'Imaging': 1.2, 'Therapy': 1.0, 'Lab Work': 0.8, 'Routine': 0.6}
cost = np.array([base_cost[d] * proc_mult[p] * (1 + 0.1 * l) * np.random.uniform(0.7, 1.3)
                  for d, p, l in zip(diagnosis, procedure, los)])
cost = np.round(cost, 2)

df = pd.DataFrame({
    'PatientID': [f'P{i:06d}' for i in range(n)],
    'Age': age, 'Gender': gender, 'Insurance': insurance,
    'Diagnosis': diagnosis, 'Procedure': procedure,
    'LengthOfStay': los, 'TotalCharge': cost
})
print(f'Dataset: {df.shape}')
print(f'Cost range: ${df["TotalCharge"].min():,.0f} — ${df["TotalCharge"].max():,.0f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nCharge stats:\n{df["TotalCharge"].describe().round(2)}')

## Cost Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['TotalCharge'].hist(bins=50, ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Distribution of Total Charges')
axes[0].set_xlabel('Charge ($)')
df['TotalCharge'].apply(np.log1p).hist(bins=50, ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('Log-Transformed Charges')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cost_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Cost by Diagnosis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
order = df.groupby('Diagnosis')['TotalCharge'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Diagnosis', y='TotalCharge', order=order, ax=ax)
ax.set_title('Charges by Diagnosis')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cost_by_diagnosis.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Median charge by diagnosis:')
print(df.groupby('Diagnosis')['TotalCharge'].median().sort_values(ascending=False).round(0))

## Cost by Procedure Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
order = df.groupby('Procedure')['TotalCharge'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Procedure', y='TotalCharge', order=order, ax=ax)
ax.set_title('Charges by Procedure Type')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cost_by_procedure.png'), dpi=100, bbox_inches='tight')
plt.show()

## Length of Stay Impact

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
los_cost = df.groupby('LengthOfStay')['TotalCharge'].mean()
los_cost.plot(ax=ax, marker='o', color='coral')
ax.set_title('Average Charge by Length of Stay')
ax.set_xlabel('Length of Stay (days)')
ax.set_ylabel('Average Charge ($)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'los_cost.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Correlation LOS vs Charge: {df["LengthOfStay"].corr(df["TotalCharge"]):.3f}')

## Insurance Type Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Insurance', y='TotalCharge', ax=ax)
ax.set_title('Charges by Insurance Type')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cost_by_insurance.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Mean charge by insurance:')
print(df.groupby('Insurance')['TotalCharge'].mean().sort_values(ascending=False).round(0))

## Top Cost Drivers Summary

In [ ]:
# Variance contribution via groupby
factors = ['Diagnosis', 'Procedure', 'Insurance', 'Gender']
total_var = df['TotalCharge'].var()
print(f'Total charge variance: ${total_var:,.0f}\n')
for f in factors:
    group_means = df.groupby(f)['TotalCharge'].mean()
    between_var = sum(len(df[df[f]==g]) * (group_means[g] - df['TotalCharge'].mean())**2 for g in group_means.index) / len(df)
    print(f'{f}: explains {between_var/total_var:.1%} of variance')

print(f'\nLengthOfStay R²: {df["LengthOfStay"].corr(df["TotalCharge"])**2:.1%}')

## Interpretation and Business Insight
- **Diagnosis** is the strongest categorical cost driver — Cancer and Heart Disease dominate
- **Procedure type** is the second strongest — Surgery is 2-3x more expensive than routine care
- **Length of Stay** has a strong positive correlation with total charge
- **Insurance type** shows modest variation — Self-Pay may reflect selection bias
- **Gender** has minimal direct impact on cost
- Combined, diagnosis + procedure + LOS explain most cost variation

## Limitations
- Synthetic data with simplified cost model
- No comorbidity data
- No readmission or outcome data
- No regional or facility variation
- No time-based cost trends

## How to Improve This Project
- Add comorbidity index (Charlson) as a cost driver
- Include readmission costs and 30-day total cost of care
- Add facility-level comparison
- Build predictive cost models
- Analyze cost per quality-adjusted outcome

## Production Considerations
- Real-time cost analytics dashboards for hospital finance
- Predictive cost models for pre-authorization
- DRG-based cost benchmarking
- Integration with clinical decision support for cost-conscious care paths

## Common Mistakes
- Comparing costs without adjusting for case mix
- Ignoring length of stay as a confounding factor
- Equating charges with actual cost-to-deliver
- Not segmenting by diagnosis when analyzing procedure costs

## Mini Challenge / Exercises
1. What is the most expensive diagnosis-procedure combination?
2. Find patients in the top 5% of charges — what diagnosis and procedure patterns dominate?
3. Calculate the cost per day of stay for each diagnosis category.
4. If LOS was reduced by 1 day on average, estimate total savings.

## Final Summary / Key Takeaways
- Diagnosis and procedure type are the dominant cost drivers
- Length of stay amplifies costs linearly
- Targeted interventions on high-cost diagnoses and surgical pathways offer the largest savings
- Insurance-based cost variation is modest compared to clinical factors
- Cost transparency requires proper case-mix adjustment